# SkyPortal Inventory Coverage

## 0. Purpose and scope

**Question.** What does the locally captured SkyPortal source inventory contain, and
which later ingestion decisions are justified by that evidence?

This notebook freezes the **2026-07-20** capture as the primary evidence. This notebook
works on a single frozen capture (2026-07-20). The SkyPortal inventory grows over time
and occasionally loses sources, so all counts below are tied to that date. By the end,
the reader knows the population definition, profile consistency, compact-file losses,
field coverage, history completeness controls, missing collections, and trigger-time
coverage.

In [1]:
from pathlib import Path
import ast
import json
import re
import sys

import pandas as pd
import yaml

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_ROOT = ROOT / "data" / "raw" / "skyportal" / "inventory"
EVIDENCE_DIR = ROOT / "notebooks" / "evidence"
JULY_STAMP = "20260720"
PROFILES = ["grandma_base", "gcn", "ep", "grb"]

print(f"Python: {sys.executable}")
print(f"Repository: {ROOT}")
print("Frozen capture: 2026-07-20")
print(f"Raw inventory root exists: {RAW_ROOT.exists()}")

Python: /home/meneses/project_astronomical/MAFORAI/.venv/bin/python
Repository: /home/meneses/project_astronomical/MAFORAI
Frozen capture: 2026-07-20
Raw inventory root exists: True


**DECISION.** Treat this notebook and its extracts as frozen evidence for the
2026-07-20 population. A later inventory must be analyzed as a separate dated capture
rather than silently replacing these counts.

## 1. What we are loading

**Question.** Which four profile queries define the source population, and can the
analysis run from committed slim extracts when raw pages are unavailable?

In [2]:
CONFIG_PATH = ROOT / "configs" / "extraction" / "skyportal.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

profile_rows = []
for profile_name in PROFILES:
    profile = config["inventory"]["profiles"][profile_name]
    template_name = profile["query_template"]
    query_params = dict(config["inventory"]["shared_query_params"][template_name])
    query_params.update(profile.get("query_params", {}))
    profile_rows.append(
        {
            "profile": profile_name,
            "max_pages": profile["max_pages"],
            "query_template": template_name,
            "query_params": json.dumps(query_params, sort_keys=True),
        }
    )
profile_queries = pd.DataFrame(profile_rows)
print(profile_queries.to_string(index=False))

run_directories = {}
for profile_name in PROFILES:
    run_directories[profile_name] = sorted(
        RAW_ROOT.glob(f"source_inventory_{profile_name}_{JULY_STAMP}_*")
    )

raw_available = all(len(matches) == 1 for matches in run_directories.values())
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)


def is_non_empty(value):
    return value is not None and value != "" and value != [] and value != {}


def classify_name(source_id):
    if re.fullmatch(r"GCN-\d{6}_\d{6}", source_id, re.IGNORECASE):
        return "gcn_internal"
    if re.fullmatch(r"EP-\d{6}_\d{6}", source_id, re.IGNORECASE):
        return "ep_internal"
    if re.fullmatch(r"GRB-\d{6}_\d{6}", source_id, re.IGNORECASE):
        return "grb_internal"
    if re.match(r"GRB", source_id, re.IGNORECASE):
        return "grb_named"
    if re.match(r"(AT|SN)\s?\d{4}", source_id, re.IGNORECASE):
        return "tns_like"
    if re.match(r"ZTF", source_id, re.IGNORECASE):
        return "ztf_like"
    return "other"


records_by_profile = {}
records_by_source = {}
if raw_available:
    for profile_name, directories in run_directories.items():
        records_by_profile[profile_name] = []
        for page_path in sorted(directories[0].glob("sources_page_*.json")):
            payload = json.loads(page_path.read_text(encoding="utf-8"))
            data = payload.get("data", {})
            records_by_profile[profile_name].extend(
                row for row in data.get("sources", []) if isinstance(row, dict)
            )
        for record in records_by_profile[profile_name]:
            source_id = record.get("id")
            if isinstance(source_id, str):
                records_by_source.setdefault(source_id, {})[profile_name] = record

    july_rows = [
        record for records in records_by_profile.values() for record in records
    ]
    all_fields = sorted({field for record in july_rows for field in record})
    field_coverage_rows = []
    for field in all_fields:
        n_present = sum(field in record for record in july_rows)
        n_non_empty = sum(
            field in record and is_non_empty(record[field]) for record in july_rows
        )
        field_coverage_rows.append(
            {
                "field": field,
                "n_present": n_present,
                "n_non_empty": n_non_empty,
                "pct_non_empty": 100.0 * n_non_empty / len(july_rows),
            }
        )
    field_coverage = pd.DataFrame(field_coverage_rows).sort_values(
        ["pct_non_empty", "field"], ascending=[False, True]
    )

    source_index_rows = []
    for source_id in sorted(records_by_source):
        profile_records = records_by_source[source_id]
        selected_profile, selected_record = max(
            profile_records.items(),
            key=lambda item: (
                sum(is_non_empty(value) for value in item[1].values()),
                item[0],
            ),
        )
        source_index_rows.append(
            {
                "source_id": source_id,
                "profiles": "|".join(sorted(profile_records)),
                "name_pattern_class": classify_name(source_id),
                "has_t0": is_non_empty(selected_record.get("t0")),
                "t0": selected_record.get("t0"),
                "n_redshift_versions": len(
                    selected_record.get("redshift_history") or []
                ),
                "n_summary_versions": len(
                    selected_record.get("summary_history") or []
                ),
                "n_classifications": len(
                    selected_record.get("classifications") or []
                ),
                "n_annotations": len(selected_record.get("annotations") or []),
                "has_alias": is_non_empty(selected_record.get("alias")),
                "has_tns_name": is_non_empty(selected_record.get("tns_name")),
                "comment_exists": bool(selected_record.get("comment_exists")),
                "photometry_exists": bool(selected_record.get("photometry_exists")),
                "spectrum_exists": bool(selected_record.get("spectrum_exists")),
                "created_at": selected_record.get("created_at"),
                "modified": selected_record.get("modified"),
                "ra": selected_record.get("ra"),
                "dec": selected_record.get("dec"),
                "redshift": selected_record.get("redshift"),
            }
        )

    source_index = pd.DataFrame(source_index_rows)
    source_index.to_csv(EVIDENCE_DIR / "01_source_index.csv", index=False)
    field_coverage.to_csv(EVIDENCE_DIR / "01_field_coverage.csv", index=False)
    print("Input path: raw July inventory pages; slim extracts were refreshed.")
else:
    source_index = pd.read_csv(EVIDENCE_DIR / "01_source_index.csv")
    field_coverage = pd.read_csv(EVIDENCE_DIR / "01_field_coverage.csv")
    records_by_profile = {}
    records_by_source = {}
    july_rows = []
    print("Input path: committed slim extracts; raw July inventory pages were unavailable.")

print(f"Source index rows: {len(source_index)}")
print(f"Field coverage rows: {len(field_coverage)}")

     profile  max_pages       query_template                                                                                                                                                                                                               query_params
grandma_base          0 enriched_recent_desc {"group_ids": "3", "includeCommentExists": "true", "includeDetectionStats": "true", "includeHosts": "true", "includePhotometryExists": "true", "includeSpectrumExists": "true", "sortBy": "saved_at", "sortOrder": "desc"}
         gcn          0 enriched_recent_desc                        {"includeCommentExists": "true", "includeDetectionStats": "true", "includePhotometryExists": "true", "includeSpectrumExists": "true", "sortBy": "saved_at", "sortOrder": "desc", "sourceID": "GCN"}
          ep          0 enriched_recent_desc                         {"includeCommentExists": "true", "includeDetectionStats": "true", "includePhotometryExists": "true", "includeSpectrumExists": "true", "sort

**DECISION.** Define the corpus population as the **union** of the `grandma_base`,
`gcn`, `ep`, and `grb` queries printed above. This definition must be stated in the
project documentation because changing any query changes the population.

## 2. Population

**Question.** How many listing records and unique source IDs are in the 2026-07-20
union, and how much profile overlap exists?

In [3]:
current_sources = source_index.copy()

if raw_available:
    population_rows = []
    for profile_name in PROFILES:
        records = records_by_profile[profile_name]
        population_rows.append(
            {
                "profile": profile_name,
                "records": len(records),
                "unique_source_ids": len({record["id"] for record in records}),
            }
        )
    population = pd.DataFrame(population_rows)
    july_record_count = sum(population["records"])
else:
    population_rows = []
    for profile_name in PROFILES:
        count = current_sources["profiles"].fillna("").str.split("|").apply(
            lambda values: profile_name in values
        ).sum()
        population_rows.append(
            {"profile": profile_name, "records": count, "unique_source_ids": count}
        )
    population = pd.DataFrame(population_rows)
    july_record_count = int(field_coverage["n_present"].max())

n_profile_overlaps = current_sources["profiles"].fillna("").str.contains(r"\|").sum()
print(population.to_string(index=False))
print(f"All July listing records: {july_record_count}")
print(f"Unique July source IDs: {len(current_sources)}")
print(f"Sources in two or more profiles: {n_profile_overlaps}")

     profile  records  unique_source_ids
grandma_base      407                407
         gcn      155                155
          ep      219                219
         grb      201                201
All July listing records: 982
Unique July source IDs: 800
Sources in two or more profiles: 182


**DECISION.** The working 2026-07-20 population is **800 unique sources** represented
by **982 listing records**; 182 sources occur in multiple profiles. All later coverage
statistics use this full union and never filter sources by name pattern.

## 3. Are profile records consistent?

**Question.** When the same source appears in multiple profiles, do key sets or the
scientific history arrays disagree?

In [4]:
if raw_available:
    overlapping_ids = sorted(
        source_id
        for source_id, profile_records
        in records_by_source.items()
        if len(profile_records) > 1
    )
    key_differences = set()
    value_differences = {
        field: 0
        for field in [
            "redshift_history",
            "summary_history",
            "classifications",
            "annotations",
        ]
    }
    examples = []
    for source_id in overlapping_ids:
        profile_records = records_by_source[source_id]
        profile_names = sorted(profile_records)
        left_name, right_name = profile_names[:2]
        left = profile_records[left_name]
        right = profile_records[right_name]
        key_differences.update(set(left) ^ set(right))
        for field in value_differences:
            if left.get(field) != right.get(field):
                value_differences[field] += 1
        if len(examples) < 3:
            examples.append(
                {
                    "source_id": source_id,
                    "profiles": f"{left_name}|{right_name}",
                    "key_difference": "|".join(sorted(set(left) ^ set(right))),
                    **{
                        f"{field}_same": left.get(field) == right.get(field)
                        for field in value_differences
                    },
                }
            )
    print(f"Overlapping sources compared: {len(overlapping_ids)}")
    print(f"Union of cross-profile key differences: {sorted(key_differences)}")
    print("Value differences across overlapping sources:")
    print(pd.Series(value_differences).to_string())
    print("Three real comparisons:")
    print(pd.DataFrame(examples).to_string(index=False))
else:
    print("UNKNOWN: slim extracts retain profile membership but not duplicate raw values.")
    print("Use the committed executed output above as the frozen comparison evidence.")

Overlapping sources compared: 182
Union of cross-profile key differences: ['host', 'host_offset']
Value differences across overlapping sources:
redshift_history    0
summary_history     0
classifications     0
annotations         0
Three real comparisons:
       source_id        profiles   key_difference  redshift_history_same  summary_history_same  classifications_same  annotations_same
EP-241103_012438 ep|grandma_base host|host_offset                   True                  True                  True              True
EP-250704_081653 ep|grandma_base host|host_offset                   True                  True                  True              True
EP-250727_073405 ep|grandma_base host|host_offset                   True                  True                  True              True


**DECISION.** Deduplicate a source ID by choosing the richest available profile record,
with `grandma_base` winning where `includeHosts=true` adds `host` and `host_offset`.
The four compared scientific arrays are value-identical across duplicate records, so
this rule preserves their content.

## 4. What the current compact file discards

**Question.** Which listing sources are absent from `gcn_grandma.json`, what kinds of
names do they have, and do they carry real data?

In [5]:
COMPACT_PATH = ROOT / "data" / "interim" / "skyportal" / "gcn_grandma.json"
compact_payload = json.loads(COMPACT_PATH.read_text(encoding="utf-8"))
compact_ids = {
    row["id"] for row in compact_payload["sources"] if isinstance(row.get("id"), str)
}
listing_ids = set(current_sources["source_id"])
missing_ids = sorted(listing_ids - compact_ids)
missing = current_sources[current_sources["source_id"].isin(missing_ids)].copy()

print(f"July listing sources: {len(listing_ids)}")
print(f"Compact sources: {len(compact_ids)}")
print(f"Listing sources absent from compact: {len(missing_ids)}")
print(f"Compact sources absent from July listing: {len(compact_ids - listing_ids)}")
print("Missing sources by name-pattern class:")
print(missing["name_pattern_class"].value_counts().to_string())

missing_flags = pd.DataFrame(
    {
        "condition": [
            "comment_exists",
            "photometry_exists",
            "spectrum_exists",
            "has_t0",
            "has_classifications",
        ],
        "sources": [
            int(missing["comment_exists"].fillna(False).astype(bool).sum()),
            int(missing["photometry_exists"].fillna(False).astype(bool).sum()),
            int(missing["spectrum_exists"].fillna(False).astype(bool).sum()),
            int(missing["has_t0"].fillna(False).astype(bool).sum()),
            int((missing["n_classifications"] > 0).sum()),
        ],
    }
)
print("Data-bearing flags among missing sources:")
print(missing_flags.to_string(index=False))
missing_has_data = (
    missing["comment_exists"].fillna(False).astype(bool)
    | missing["photometry_exists"].fillna(False).astype(bool)
    | missing["spectrum_exists"].fillna(False).astype(bool)
    | missing["has_t0"].fillna(False).astype(bool)
    | (missing["n_classifications"] > 0)
)
print(
    f"Missing sources with at least one checked data-bearing flag: "
    f"{int(missing_has_data.sum())}"
)
print(
    f"Missing sources without any checked data-bearing flag: "
    f"{int((~missing_has_data).sum())}"
)
print("Ten real missing source IDs:")
print(
    missing[
        [
            "source_id",
            "name_pattern_class",
            "comment_exists",
            "photometry_exists",
            "spectrum_exists",
            "has_t0",
            "n_classifications",
        ]
    ].head(10).to_string(index=False)
)

selection_path = (
    ROOT / "src" / "skyportal_corpus" / "extraction" / "source_selection.py"
)
selection_source = selection_path.read_text(encoding="utf-8")
selection_tree = ast.parse(selection_source)
selection_node = next(
    node
    for node in selection_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "build_gcn_grandma_event"
)
selection_lines = selection_source.splitlines()
print("Filtering code from build_gcn_grandma_event() (verbatim):")
print(
    "\n".join(
        selection_lines[selection_node.lineno - 1 : selection_node.end_lineno]
    )
)

July listing sources: 800
Compact sources: 576
Listing sources absent from compact: 224
Compact sources absent from July listing: 0
Missing sources by name-pattern class:
name_pattern_class
other       186
ztf_like     25
tns_like     13
Data-bearing flags among missing sources:
          condition  sources
     comment_exists      160
  photometry_exists      114
    spectrum_exists        1
             has_t0        6
has_classifications      106
Missing sources with at least one checked data-bearing flag: 196
Missing sources without any checked data-bearing flag: 28
Ten real missing source IDs:
source_id name_pattern_class  comment_exists  photometry_exists  spectrum_exists  has_t0  n_classifications
 2024abfo              other           False               True            False   False                  0
  2024hdk              other            True              False            False   False                  0
  2024hdr              other            True              False       

**DECISION.** The 224 listing sources excluded by the compact GCN-derived-name filter
belong in the corpus population: 196 carry at least one checked data-bearing flag, and
28 have none of those flags but remain members of the declared four-profile union. A
new emitter must retain all listing sources and must not reuse this name-based
population filter.

## 5. Field coverage

**Question.** Which listing fields are always, partially, or never populated across all
982 July records?

In [6]:
print(field_coverage.to_string(index=False, formatters={
    "pct_non_empty": lambda value: f"{value:.2f}%"
}))

always_fields = field_coverage.loc[
    field_coverage["pct_non_empty"] == 100, "field"
].tolist()
partial_fields = field_coverage.loc[
    field_coverage["pct_non_empty"].between(0, 100, inclusive="neither"), "field"
].tolist()
never_fields = field_coverage.loc[
    field_coverage["pct_non_empty"] == 0, "field"
].tolist()

print(f"Always populated ({len(always_fields)}): {always_fields}")
print(f"Partially populated ({len(partial_fields)}): {partial_fields}")
print(f"Never populated ({len(never_fields)}): {never_fields}")

                    field  n_present  n_non_empty pct_non_empty
           comment_exists        982          982       100.00%
               created_at        982          982       100.00%
                      dec        982          982       100.00%
                  gal_lat        982          982       100.00%
                  gal_lon        982          982       100.00%
                   groups        982          982       100.00%
                  healpix        982          982       100.00%
                       id        982          982       100.00%
             internal_key        982          982       100.00%
                  is_roid        982          982       100.00%
                 modified        982          982       100.00%
                   offset        982          982       100.00%
        photometry_exists        982          982       100.00%
                       ra        982          982       100.00%
          spectrum_exists        982    

**DECISION.** Preserve as ledger fact inputs the source identity and timing fields
`id`, `alias`, `tns_name`, `created_at`, `modified`, `t0`; coordinates and redshift
fields `ra`, `dec`, `redshift`, `redshift_error`, `redshift_origin`,
`redshift_history`; and structured scientific histories `summary_history`,
`classifications`, `annotations`, `gcn_crossmatch`, `host`, `host_offset`, and
`photstats`. Keep the remaining non-empty listing fields as raw context until the
ledger schema is designed. Drop the empirically empty fields `altdata`, `dec_dis`,
`dec_err`, `detect_photometry_count`, `dist_nearest_source`,
`e_mag_nearest_source`, `mag_nearest_source`, `mpc_name`, `ra_dis`, `ra_err`, and
`score` from the slim extract.

## 6. Is the listing truncated?

**Question.** Do listing-level history arrays contain fewer versions than were
previously observed from the per-source endpoint?

In [7]:
controls = pd.DataFrame(
    [
        ("2025aji", 4, 8),
        ("GRB241030", 1, 6),
        ("2026owq", 2, 44),
        ("EP-260623_025405", 2, 24),
    ],
    columns=[
        "source_id",
        "expected_redshift_versions",
        "expected_summary_versions",
    ],
)
control_rows = controls.merge(
    source_index[
        ["source_id", "n_redshift_versions", "n_summary_versions"]
    ],
    on="source_id",
    how="left",
)
control_rows["redshift_matches"] = (
    control_rows["expected_redshift_versions"]
    == control_rows["n_redshift_versions"]
)
control_rows["summary_matches"] = (
    control_rows["expected_summary_versions"]
    == control_rows["n_summary_versions"]
)
print(control_rows.to_string(index=False))

       source_id  expected_redshift_versions  expected_summary_versions  n_redshift_versions  n_summary_versions  redshift_matches  summary_matches
         2025aji                           4                          8                    4                   8              True             True
       GRB241030                           1                          6                    1                   6              True             True
         2026owq                           2                         44                    2                  44              True             True
EP-260623_025405                           2                         24                    2                  24              True             True


**DECISION.** The listing matches the prior per-source history counts for all four
controls. Source-level redshift and summary histories therefore do not require another
per-source download on the evidence available here; this conclusion is limited to
these controls.

## 7. What the listing does not contain

**Question.** Which row-level collections are absent, and what is the minimum request
budget to fetch them for the full population?

In [8]:
coverage_fields = set(field_coverage["field"])
collections = ["comments", "photometry", "spectra", "followup_requests"]
collection_check = pd.DataFrame(
    {
        "collection": collections,
        "top_level_field_present": [
            field in coverage_fields for field in collections
        ],
    }
)
print(collection_check.to_string(index=False))

flag_rows = pd.DataFrame(
    {
        "field": ["comment_exists", "photometry_exists", "spectrum_exists"],
        "meaning": [
            "boolean flag, not comment rows",
            "boolean flag, not photometry rows",
            "boolean flag, not spectrum rows",
        ],
        "n_non_empty": [
            int(field_coverage.set_index("field").loc[field, "n_non_empty"])
            for field in ["comment_exists", "photometry_exists", "spectrum_exists"]
        ],
    }
)
print(flag_rows.to_string(index=False))

n_sources = len(current_sources)
n_collections = len(collections)
n_calls = n_sources * n_collections
minimum_seconds = n_calls * 0.5
print(f"Sources x collections: {n_sources} x {n_collections} = {n_calls} calls")
print(f"Minimum pacing time at 0.5 s: {minimum_seconds:.1f} s ({minimum_seconds / 60:.1f} min)")

       collection  top_level_field_present
         comments                    False
       photometry                    False
          spectra                    False
followup_requests                    False
            field                           meaning  n_non_empty
   comment_exists    boolean flag, not comment rows          982
photometry_exists boolean flag, not photometry rows          982
  spectrum_exists   boolean flag, not spectrum rows          982
Sources x collections: 800 x 4 = 3200 calls
Minimum pacing time at 0.5 s: 1600.0 s (26.7 min)


**DECISION.** A per-source download stage is required for comments, photometry,
spectra, and follow-up requests. Its initial scope is 3,200 collection requests for
800 sources, and it needs checkpoint/resume because a paced multi-request run cannot
be assumed to finish reliably in one uninterrupted process.

## 8. A measured temporal-anchor ladder

### 8.1. Is the missing `t0` concentrated in events we care about?

**Question.** Among sources without `t0`, how many already carry comments, photometry,
spectra, or classifications that make temporal reconstruction worthwhile?

In [9]:
source_index_base_columns = [
    column
    for column in source_index.columns
    if column not in {
        "anchor_type",
        "t0_from_id",
        "t0_source",
        "t0_uncertainty_hours",
        "tier_status",
    }
]
current_t0 = source_index.copy()
current_t0["has_t0"] = current_t0["has_t0"].fillna(False).astype(bool)
for flag in ["comment_exists", "photometry_exists", "spectrum_exists"]:
    current_t0[flag] = current_t0[flag].fillna(False).astype(bool)
current_t0["has_classifications"] = current_t0["n_classifications"].gt(0)

flag_rows = []
for flag in [
    "comment_exists",
    "photometry_exists",
    "spectrum_exists",
    "has_classifications",
]:
    flag_rows.append(
        {
            "data_bearing_flag": flag,
            "without_t0": int(current_t0.loc[~current_t0["has_t0"], flag].sum()),
            "with_t0": int(current_t0.loc[current_t0["has_t0"], flag].sum()),
            "all_sources": int(current_t0[flag].sum()),
        }
    )
print(pd.DataFrame(flag_rows).to_string(index=False))

name_t0 = pd.crosstab(
    current_t0["name_pattern_class"], current_t0["has_t0"]
).rename(columns={False: "without_t0", True: "with_t0"})
for column in ["without_t0", "with_t0"]:
    if column not in name_t0:
        name_t0[column] = 0
name_t0["all_sources"] = name_t0["without_t0"] + name_t0["with_t0"]
name_t0["pct_with_t0"] = 100.0 * name_t0["with_t0"] / name_t0["all_sources"]
print(name_t0.reset_index().to_string(index=False, formatters={
    "pct_with_t0": lambda value: f"{value:.2f}%"
}))

missing_t0 = current_t0.loc[~current_t0["has_t0"]]
missing_t0_has_data = missing_t0[
    [
        "comment_exists",
        "photometry_exists",
        "spectrum_exists",
        "has_classifications",
    ]
].any(axis=1)
print(
    "Sources without t0 carrying at least one data-bearing flag: "
    f"{int(missing_t0_has_data.sum())}/{len(missing_t0)}"
)

  data_bearing_flag  without_t0  with_t0  all_sources
     comment_exists         233      118          351
  photometry_exists         138      102          240
    spectrum_exists           0        1            1
has_classifications         192       59          251
name_pattern_class  without_t0  with_t0  all_sources pct_with_t0
       ep_internal         189        5          194       2.58%
      gcn_internal         102       53          155      34.19%
      grb_internal          74       31          105      29.52%
         grb_named          67       27           94      28.72%
             other         203       11          214       5.14%
          tns_like          13        0           13       0.00%
          ztf_like          24        1           25       4.00%
Sources without t0 carrying at least one data-bearing flag: 296/672


**FINDING.** Of the 672 sources without `t0`, **296** already carry at least one data-bearing flag, so temporal reconstruction concerns a substantial active subset.

### 8.2. Two kinds of anchor

A trigger-class source has a physical trigger instant. A survey-discovered transient
has no trigger instant: its anchor is a first-detection epoch, which is an upper bound
on the explosion time. Consequently, delta-t denotes time since trigger in the first
group and time since first detection in the second; the two quantities must not be
interpreted as equivalent.

In [10]:
trigger_name_classes = {
    "gcn_internal",
    "ep_internal",
    "grb_internal",
    "grb_named",
}
current_t0["anchor_type"] = "first_detection"
current_t0.loc[
    current_t0["name_pattern_class"].isin(trigger_name_classes),
    "anchor_type",
] = "trigger"

anchor_coverage = (
    current_t0.groupby("anchor_type")["has_t0"]
    .agg(sources="size", sources_with_t0="sum")
    .reset_index()
)
anchor_coverage["sources_without_t0"] = (
    anchor_coverage["sources"] - anchor_coverage["sources_with_t0"]
)
anchor_coverage["pct_with_t0"] = (
    100.0 * anchor_coverage["sources_with_t0"] / anchor_coverage["sources"]
)
print(anchor_coverage.to_string(index=False, formatters={
    "pct_with_t0": lambda value: f"{value:.2f}%"
}))

    anchor_type  sources  sources_with_t0  sources_without_t0 pct_with_t0
first_detection      252               12                 240       4.76%
        trigger      548              116                 432      21.17%


**FINDING.** The population separates into **548 trigger anchors** and **252 first-detection anchors**, with real-`t0` coverage of **21.17%** and **4.76%**, respectively.

### 8.3. Does the internal source ID encode the trigger time?

Internal IDs matching `PREFIX-YYMMDD_HHMMSS` are parsed as UTC timestamps. Validation
uses only sources that also carry a real SkyPortal `t0`; signed offsets are
`timestamp_from_id - real_t0`. Threshold counts use absolute offsets.

In [11]:
id_parts = current_t0["source_id"].str.extract(
    r"^(?P<id_prefix>GCN|EP|GRB)-(?P<id_date>\d{6})_(?P<id_clock>\d{6})$",
    flags=re.IGNORECASE,
)
id_timestamp = pd.to_datetime(
    id_parts["id_date"].fillna("") + id_parts["id_clock"].fillna(""),
    format="%y%m%d%H%M%S",
    errors="coerce",
    utc=True,
)
current_t0["id_prefix"] = id_parts["id_prefix"].str.upper()
current_t0["t0_from_id"] = (
    (id_timestamp - pd.Timestamp("1858-11-17", tz="UTC"))
    / pd.Timedelta(days=1)
)

id_pattern_matches = id_parts["id_prefix"].notna()
id_parse_failures = current_t0.loc[
    id_pattern_matches & current_t0["t0_from_id"].isna(), "source_id"
]
print(f"Internal IDs matching the pattern: {int(id_pattern_matches.sum())}")
print(f"Parsed successfully: {int(current_t0['t0_from_id'].notna().sum())}")
print(f"Parse failures: {len(id_parse_failures)}")
if len(id_parse_failures):
    print(id_parse_failures.head(3).to_string(index=False))

id_validation = current_t0.loc[
    current_t0["has_t0"] & current_t0["t0_from_id"].notna(),
    ["source_id", "id_prefix", "t0", "t0_from_id"],
].copy()
id_validation["delta_seconds"] = (
    pd.to_numeric(id_validation["t0_from_id"])
    - pd.to_numeric(id_validation["t0"])
) * 86400.0
id_validation["abs_delta_seconds"] = id_validation["delta_seconds"].abs()
print(f"Validation set size: {len(id_validation)}")
if len(id_validation) < 10:
    print("Validation is indicative only because the set contains fewer than 10 sources.")

overall_id_offset = pd.DataFrame(
    [
        {
            "n": len(id_validation),
            "min_s": id_validation["delta_seconds"].min(),
            "p25_s": id_validation["delta_seconds"].quantile(0.25),
            "median_s": id_validation["delta_seconds"].median(),
            "p75_s": id_validation["delta_seconds"].quantile(0.75),
            "p95_s": id_validation["delta_seconds"].quantile(0.95),
            "max_s": id_validation["delta_seconds"].max(),
            "abs_over_5_min": int((id_validation["abs_delta_seconds"] > 300).sum()),
            "abs_over_1_h": int((id_validation["abs_delta_seconds"] > 3600).sum()),
            "abs_over_6_h": int((id_validation["abs_delta_seconds"] > 21600).sum()),
        }
    ]
)
print("All internal prefixes:")
print(overall_id_offset.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

prefix_rows = []
for prefix, group in id_validation.groupby("id_prefix", sort=True):
    prefix_rows.append(
        {
            "prefix": prefix,
            "n": len(group),
            "min_s": group["delta_seconds"].min(),
            "p25_s": group["delta_seconds"].quantile(0.25),
            "median_s": group["delta_seconds"].median(),
            "p75_s": group["delta_seconds"].quantile(0.75),
            "p95_s": group["delta_seconds"].quantile(0.95),
            "max_s": group["delta_seconds"].max(),
            "abs_over_5_min": int((group["abs_delta_seconds"] > 300).sum()),
            "abs_over_1_h": int((group["abs_delta_seconds"] > 3600).sum()),
            "abs_over_6_h": int((group["abs_delta_seconds"] > 21600).sum()),
            "assessment": "indicative only" if len(group) < 10 else "measured",
        }
    )
prefix_id_offsets = pd.DataFrame(prefix_rows)
print("By prefix:")
print(prefix_id_offsets.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

large_id_offsets = id_validation.loc[
    id_validation["abs_delta_seconds"] > 21600,
    ["source_id", "id_prefix", "delta_seconds"],
]
if not large_id_offsets.empty:
    print("Offsets exceeding 6 hours:")
    print(large_id_offsets.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

ep_control = id_validation.loc[
    id_validation["source_id"].eq("EP-260623_025405")
]
print("EP-260623_025405 control:")
print(ep_control.to_string(index=False, float_format=lambda value: f"{value:.8f}"))

Internal IDs matching the pattern: 454
Parsed successfully: 454
Parse failures: 0
Validation set size: 89


All internal prefixes:
 n     min_s  p25_s  median_s  p75_s  p95_s       max_s  abs_over_5_min  abs_over_1_h  abs_over_6_h
89 -1310.000 -7.000    -0.000  1.000 36.520 1034457.000               5             1             1
By prefix:
prefix  n     min_s   p25_s  median_s  p75_s   p95_s       max_s  abs_over_5_min  abs_over_1_h  abs_over_6_h      assessment
    EP  5    -0.000   0.000    26.000 98.000 794.944     969.180               1             0             0 indicative only
   GCN 53 -1310.000 -20.000    -5.000  0.000  14.920    1221.374               3             0             0        measured
   GRB 31    -0.458  -0.000     0.000  1.000  13.250 1034457.000               1             1             1        measured
Offsets exceeding 6 hours:
        source_id id_prefix  delta_seconds
GRB-250329_041752       GRB    1034457.000
EP-260623_025405 control:
       source_id id_prefix             t0     t0_from_id  delta_seconds  abs_delta_seconds
EP-260623_025405        EP 61214.119

**FINDING.** All **454** matching internal IDs parse successfully. GCN has 53 direct controls, EP has only 5, and GRB contains one directly measured **287.349 h** failure.

### 8.4. How bad is `created_at` as a proxy?

This comparison also uses only sources with real `t0`. `created_at - t0` measures how
late the source row entered SkyPortal, not uncertainty in a physical trigger or
detection epoch.

In [12]:
created_at_timestamp = pd.to_datetime(
    current_t0["created_at"], errors="coerce", utc=True
)
created_at_mjd = (
    (created_at_timestamp - pd.Timestamp("1858-11-17", tz="UTC"))
    / pd.Timedelta(days=1)
)
created_validation = current_t0.loc[
    current_t0["has_t0"] & created_at_timestamp.notna(),
    ["source_id", "anchor_type", "t0"],
].copy()
created_validation["delta_hours"] = (
    created_at_mjd.loc[created_validation.index]
    - pd.to_numeric(created_validation["t0"])
) * 24.0

created_rows = []
for anchor_label, group in [
    ("all", created_validation),
    *list(created_validation.groupby("anchor_type", sort=True)),
]:
    created_rows.append(
        {
            "anchor_type": anchor_label,
            "n": len(group),
            "min_h": group["delta_hours"].min(),
            "median_h": group["delta_hours"].median(),
            "p90_h": group["delta_hours"].quantile(0.90),
            "max_h": group["delta_hours"].max(),
        }
    )
created_at_offsets = pd.DataFrame(created_rows)
print(created_at_offsets.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

    anchor_type   n  min_h  median_h   p90_h     max_h
            all 128  0.005     0.171  24.653 17572.145
first_detection  12  1.091    16.036 178.095 17572.145
        trigger 116  0.005     0.116  10.294   349.538


**FINDING.** `created_at` is a knowledge-time floor, not a physical anchor: its maximum observed lag is **349.538 h** for trigger sources and **17,572.145 h** for first-detection sources.

### 8.5. Does the internal ID's timestamp reduce to `created_at`?

Section 8.3 validated the internal-ID timestamp against real `t0` on 89 sources, but
only 5 of those are EP, while 189 EP sources depend on that rule as their only
non-`created_at` fallback.

This subsection tests the full 454-source population of parseable IDs, then feeds the
measured diagnostic into the tier statuses in 8.6 and the decision in 8.7. Section
8.5.5 establishes the diagnostic's central limitation: closeness between the ID and
`created_at` cannot expose a record that was itself created late.

#### 8.5.1. `id_timestamp` versus `created_at`, all parseable IDs

Sign convention: `delta_id_created_s = created_at - id_timestamp`, in seconds. A
positive value means the SkyPortal row was created after the moment encoded in the
ID; this is the expected sign, since the row can only be created once the alert that
seeds the ID has been received.


In [13]:
current_t0["id_timestamp"] = id_timestamp
current_t0["delta_id_created_s"] = (created_at_timestamp - id_timestamp).dt.total_seconds()

parseable_id_mask = current_t0["id_timestamp"].notna()
delta_all = current_t0.loc[parseable_id_mask, "delta_id_created_s"]

overall_delta = pd.DataFrame(
    [
        {
            "n": len(delta_all),
            "min_s": delta_all.min(),
            "p05_s": delta_all.quantile(0.05),
            "p25_s": delta_all.quantile(0.25),
            "median_s": delta_all.median(),
            "p75_s": delta_all.quantile(0.75),
            "p95_s": delta_all.quantile(0.95),
            "max_s": delta_all.max(),
            "abs_over_5_min": int((delta_all.abs() > 300).sum()),
            "abs_over_1_h": int((delta_all.abs() > 3600).sum()),
            "abs_over_1_day": int((delta_all.abs() > 86400).sum()),
        }
    ]
)
print(f"Parseable IDs: {int(parseable_id_mask.sum())}")
print(overall_delta.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


Parseable IDs: 454
  n      min_s  p05_s   p25_s  median_s    p75_s     p95_s      max_s  abs_over_5_min  abs_over_1_h  abs_over_1_day
454 -56635.904 41.866 241.434   488.450 1521.501 42907.111 705466.738             315            65              12


#### 8.5.2. Same, broken down by prefix

The bulk distributions are comparable, but the tails are not: GCN has a p95 of
**1.0 h**, while EP has a p95 of **17.5 h**, an approximately **18x** difference.
The over-one-day rate is **0.65%** for GCN and **3.09%** for EP. Fisher's exact
**p = 0.14** fails to resolve that rate difference at this sample size; it is not
evidence that the two rates are equal.

In [14]:
import math

sub = current_t0.loc[parseable_id_mask].copy()
sub["abs_delta"] = sub["delta_id_created_s"].abs()

prefix_delta_rows = []
for prefix in sorted(sub["id_prefix"].dropna().unique()):
    group = sub.loc[sub["id_prefix"] == prefix, "delta_id_created_s"]
    abs_group = group.abs()
    prefix_delta_rows.append(
        {
            "prefix": prefix,
            "n": len(group),
            "min_s": group.min(),
            "p05_s": group.quantile(0.05),
            "p25_s": group.quantile(0.25),
            "median_s": group.median(),
            "p75_s": group.quantile(0.75),
            "p95_s": group.quantile(0.95),
            "max_s": group.max(),
            "abs_over_5_min": int((abs_group > 300).sum()),
            "abs_over_1_h": int((abs_group > 3600).sum()),
            "abs_over_1_day": int((abs_group > 86400).sum()),
        }
    )
prefix_delta = pd.DataFrame(prefix_delta_rows)
print(prefix_delta.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


def fisher_exact_two_sided(a, b, c, d):
    """Two-sided Fisher exact test p-value for a 2x2 table, no scipy dependency."""
    n, row1, row2, col1 = a + b + c + d, a + b, c + d, a + c

    def hyper_p(x):
        return (math.comb(row1, x) * math.comb(row2, col1 - x)) / math.comb(n, col1)

    observed_p = hyper_p(a)
    lo, hi = max(0, col1 - row2), min(row1, col1)
    return sum(hyper_p(x) for x in range(lo, hi + 1) if hyper_p(x) <= observed_p * 1.0000001)


ep_row = prefix_delta.loc[prefix_delta["prefix"] == "EP"].iloc[0]
gcn_row = prefix_delta.loc[prefix_delta["prefix"] == "GCN"].iloc[0]
tail_rate_p_value = fisher_exact_two_sided(
    int(ep_row["abs_over_1_day"]), int(ep_row["n"] - ep_row["abs_over_1_day"]),
    int(gcn_row["abs_over_1_day"]), int(gcn_row["n"] - gcn_row["abs_over_1_day"]),
)
print(
    f"\nEP over-1-day rate: {int(ep_row['abs_over_1_day'])}/{int(ep_row['n'])} "
    f"({100 * ep_row['abs_over_1_day'] / ep_row['n']:.2f}%); "
    f"GCN over-1-day rate: {int(gcn_row['abs_over_1_day'])}/{int(gcn_row['n'])} "
    f"({100 * gcn_row['abs_over_1_day'] / gcn_row['n']:.2f}%)"
)
print(f"Two-sided Fisher exact test on that difference: p = {tail_rate_p_value:.4f}")
print(
    f"Bulk gap: EP median {ep_row['median_s']:.1f} s ({ep_row['median_s'] / 60:.2f} min) vs "
    f"GCN median {gcn_row['median_s']:.1f} s ({gcn_row['median_s'] / 60:.2f} min)"
)
tail_rate_difference_resolved = tail_rate_p_value < 0.05
print(
    "Fisher test resolves the EP-GCN tail-rate difference at p < 0.05: "
    f"{tail_rate_difference_resolved}"
)
print(
    "A non-significant result does not establish equal tail rates; the observed "
    "EP tail remains materially heavier."
)


prefix   n      min_s   p05_s   p25_s  median_s    p75_s     p95_s      max_s  abs_over_5_min  abs_over_1_h  abs_over_1_day
    EP 194 -56635.904 264.225 347.624   679.580 3529.435 62988.860 610366.094             171            50               6
   GCN 155     35.709  73.140 413.299   845.206 1358.310  3500.132 478207.473             120             8               1
   GRB 105     31.540  35.734  45.299    85.965  233.556  7296.685 705466.738              24             7               5

EP over-1-day rate: 6/194 (3.09%); GCN over-1-day rate: 1/155 (0.65%)
Two-sided Fisher exact test on that difference: p = 0.1377
Bulk gap: EP median 679.6 s (11.33 min) vs GCN median 845.2 s (14.09 min)
Fisher test resolves the EP-GCN tail-rate difference at p < 0.05: False
A non-significant result does not establish equal tail rates; the observed EP tail remains materially heavier.


#### 8.5.3. Three-way comparison on the 89 validated sources

For sources with both a real `t0` and a parseable ID: `id_minus_t0_s`,
`created_minus_t0_s`, and `created_minus_id_s`. Section 8.3 measured `id_minus_t0`
median around 0 s; section 8.4 measured `created_at - t0` median around 0.116 h
(~7 min) for trigger sources. If the ID were simply a copy of `created_at`, the two
would be equally tight to `t0`.


In [15]:
t0_mjd = pd.to_numeric(current_t0["t0"], errors="coerce")
t0_timestamp = pd.Timestamp("1858-11-17", tz="UTC") + pd.to_timedelta(t0_mjd, unit="D")

three_way = current_t0.loc[current_t0["has_t0"] & current_t0["id_timestamp"].notna()].copy()
three_way["t0_timestamp"] = t0_timestamp.loc[three_way.index]
three_way["id_minus_t0_s"] = (three_way["id_timestamp"] - three_way["t0_timestamp"]).dt.total_seconds()
three_way["created_minus_t0_s"] = (
    created_at_timestamp.loc[three_way.index] - three_way["t0_timestamp"]
).dt.total_seconds()
three_way["created_minus_id_s"] = three_way["delta_id_created_s"]

print(f"Three-way validation set: {len(three_way)} sources")
print(
    three_way[
        [
            "source_id", "id_prefix", "t0_timestamp", "id_timestamp", "created_at",
            "id_minus_t0_s", "created_minus_t0_s", "created_minus_id_s",
        ]
    ].sort_values("source_id").to_string(index=False)
)

three_way_summary_rows = []
for label, group in [("all", three_way), *list(three_way.groupby("id_prefix", sort=True))]:
    three_way_summary_rows.append(
        {
            "prefix": label,
            "n": len(group),
            "median_abs_id_minus_t0_s": group["id_minus_t0_s"].abs().median(),
            "median_abs_created_minus_t0_s": group["created_minus_t0_s"].abs().median(),
            "median_created_minus_id_s": group["created_minus_id_s"].median(),
        }
    )
three_way_summary = pd.DataFrame(three_way_summary_rows)
print()
print(three_way_summary.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

all_row = three_way_summary.loc[three_way_summary["prefix"] == "all"].iloc[0]
tighter = (
    "id_timestamp"
    if all_row["median_abs_id_minus_t0_s"] < all_row["median_abs_created_minus_t0_s"]
    else "created_at"
)
ratio = all_row["median_abs_created_minus_t0_s"] / all_row["median_abs_id_minus_t0_s"]
print(
    f"\n{tighter} is tighter to real t0 on the median absolute offset "
    f"({all_row['median_abs_id_minus_t0_s']:.1f} s vs {all_row['median_abs_created_minus_t0_s']:.1f} s), "
    f"a factor of {ratio:.1f}x. The ID is therefore not a copy of created_at."
)


Three-way validation set: 89 sources
        source_id id_prefix                        t0_timestamp              id_timestamp                 created_at  id_minus_t0_s  created_minus_t0_s  created_minus_id_s
 EP-250704_081653        EP 2025-07-04 08:16:26.999616120+00:00 2025-07-04 08:16:53+00:00 2025-07-04T08:27:34.649624   2.600038e+01        6.676500e+02          641.649624
 EP-250806_091820        EP 2025-08-06 09:02:10.819999916+00:00 2025-08-06 09:18:20+00:00 2025-08-06T09:43:30.038494   9.691800e+02        2.479218e+03         1510.038494
 EP-260623_025405        EP 2026-06-23 02:52:26.999616241+00:00 2026-06-23 02:54:05+00:00 2026-06-23T02:59:22.577551   9.800038e+01        4.155779e+02          317.577551
 EP-260703_031357        EP 2026-07-03 03:13:57.000000146+00:00 2026-07-03 03:13:57+00:00 2026-07-03T12:26:37.098817  -1.460000e-07        3.316010e+04        33160.098817
 EP-260707_164524        EP 2026-07-07 16:45:23.999616267+00:00 2026-07-07 16:45:24+00:00 2026-07-07T17

#### 8.5.4. The GRB outlier, three ways


In [16]:
grb_outlier = current_t0.loc[current_t0["source_id"] == "GRB-250329_041752"].copy()
grb_outlier["t0_timestamp"] = t0_timestamp.loc[grb_outlier.index]
print(
    grb_outlier[["source_id", "t0_timestamp", "id_timestamp", "created_at"]].to_string(index=False)
)

outlier_id_minus_t0 = (
    grb_outlier["id_timestamp"].iloc[0] - grb_outlier["t0_timestamp"].iloc[0]
).total_seconds()
outlier_created_minus_t0 = (
    pd.to_datetime(grb_outlier["created_at"].iloc[0], utc=True) - grb_outlier["t0_timestamp"].iloc[0]
).total_seconds()
outlier_created_minus_id = float(grb_outlier["delta_id_created_s"].iloc[0])
print(
    f"\nid_minus_t0_s = {outlier_id_minus_t0:.1f} ({outlier_id_minus_t0 / 3600:.3f} h)\n"
    f"created_minus_t0_s = {outlier_created_minus_t0:.1f} ({outlier_created_minus_t0 / 3600:.3f} h)\n"
    f"created_minus_id_s = {outlier_created_minus_id:.1f} ({outlier_created_minus_id / 60:.2f} min)"
)
print(
    "\ncreated_at is ALSO ~"
    f"{outlier_created_minus_t0 / 3600:.0f} h after t0: the whole record was made late, "
    "not just the ID field. id_timestamp and created_at stay close to each other "
    f"({outlier_created_minus_id:.0f} s apart) throughout."
)

p95_by_prefix = sub.groupby("id_prefix")["abs_delta"].quantile(0.95)
sub["p95_for_own_prefix"] = sub["id_prefix"].map(p95_by_prefix)
exceeders = sub.loc[sub["abs_delta"] > sub["p95_for_own_prefix"]]
print(f"\nSources exceeding their own prefix's p95 |delta_id_created_s|: {len(exceeders)}")
print(
    exceeders["id_prefix"].value_counts().rename_axis("prefix").reset_index(name="n").to_string(index=False)
)
print(f"Without an alias: {int((~exceeders['has_alias']).sum())}/{len(exceeders)}")
rest = sub.loc[~sub.index.isin(exceeders.index)]
print(f"Without an alias among the rest: {int((~rest['has_alias']).sum())}/{len(rest)}")
print(
    "\nExceeders are proportionally similar across prefixes (roughly each prefix's own "
    "top ~5%, by construction) and share no trait beyond usually lacking an alias, which "
    "is also the majority pattern in the population at large."
)


        source_id                        t0_timestamp              id_timestamp                 created_at
GRB-250329_041752 2025-03-17 04:56:55.000031775+00:00 2025-03-29 04:17:52+00:00 2025-03-29T04:18:54.766979

id_minus_t0_s = 1034457.0 (287.349 h)
created_minus_t0_s = 1034519.8 (287.367 h)
created_minus_id_s = 62.8 (1.05 min)

created_at is ALSO ~287 h after t0: the whole record was made late, not just the ID field. id_timestamp and created_at stay close to each other (63 s apart) throughout.

Sources exceeding their own prefix's p95 |delta_id_created_s|: 24
prefix  n
    EP 10
   GCN  8
   GRB  6
Without an alias: 23/24
Without an alias among the rest: 342/430

Exceeders are proportionally similar across prefixes (roughly each prefix's own top ~5%, by construction) and share no trait beyond usually lacking an alias, which is also the majority pattern in the population at large.


#### 8.5.5. Can a late record be detected without a real `t0`?

**NO.** The GRB outlier in 8.5.4 shows `id_timestamp` and `created_at` moving
together even when both are ~287 h after the real trigger: `created_minus_id_s` for
that source is a completely unremarkable ~63 s, indistinguishable from a normal
ingestion delay. A late record is therefore invisible to any check built only from
the listing's own `source_id`/`created_at` fields -- detection needs an external,
independent time signal, either a first-photometry timestamp or a GCN circular
`TRIGGER_TIME` (both pending tiers in the ladder below), not more comparisons within
the listing itself.


In [17]:
real_t0_n = int(current_t0["has_t0"].sum())
gcn_fallback_n = int((~current_t0["has_t0"] & current_t0["id_prefix"].eq("GCN")).sum())
ep_fallback_n = int((~current_t0["has_t0"] & current_t0["id_prefix"].eq("EP")).sum())
grb_fallback_n = int((~current_t0["has_t0"] & current_t0["id_prefix"].eq("GRB")).sum())

phase_matching_n = real_t0_n + gcn_fallback_n
provisional_n = ep_fallback_n
dossier_only_n = 800 - phase_matching_n - provisional_n

status_summary = pd.DataFrame(
    [
        {"tier": "SkyPortal t0", "n": real_t0_n, "tier_status": "phase_matching"},
        {"tier": "GCN internal ID", "n": gcn_fallback_n, "tier_status": "phase_matching"},
        {"tier": "EP internal ID", "n": ep_fallback_n, "tier_status": "provisional"},
        {"tier": "GRB internal ID", "n": grb_fallback_n, "tier_status": "dossier_only"},
    ]
)
print(status_summary.to_string(index=False))
print(f"\nPhase matching: {phase_matching_n}/800")
print(f"Provisional: {provisional_n}/800")
print(f"Dossier-only: {dossier_only_n}/800")

n_spectrum_exists = int(current_t0["spectrum_exists"].sum())
print(f"\nspectrum_exists true: {n_spectrum_exists}/800 sources (residual fact type)")
if july_rows:
    n_annotations_nonempty_records = sum(
        1 for record in july_rows if record.get("annotations") not in (None, "", [], {})
    )
    print(
        f"non-empty annotations: {n_annotations_nonempty_records}/{len(july_rows)} raw "
        "listing records (residual fact type)"
    )
else:
    print("non-empty annotations: raw listing records unavailable in this run; see the committed extract")

           tier   n    tier_status
   SkyPortal t0 128 phase_matching
GCN internal ID 102 phase_matching
 EP internal ID 189    provisional
GRB internal ID  74   dossier_only

Phase matching: 230/800
Provisional: 189/800
Dossier-only: 381/800

spectrum_exists true: 1/800 sources (residual fact type)
non-empty annotations: 3/982 raw listing records (residual fact type)


**FINDING.** The internal timestamp is not a copy of `created_at`, but the two move together when an entire record is created late. The GCN evidence supports phase matching; EP remains provisional because its five direct controls cannot exclude that hidden failure mode, and GRB has one observed catastrophic failure.

### 8.6. The measured ladder

**PHASE MATCHING** means the anchor is accepted for temporal phase comparisons.
**PROVISIONAL** means the anchor is retained but excluded from phase matching pending
a named validation. **DOSSIER ONLY** means it provides ordering or context but is not
a physical phase anchor. Pending tiers are not yet applied.

In [18]:
missing_real_t0 = ~current_t0["has_t0"]
id_fallback = missing_real_t0 & current_t0["t0_from_id"].notna()
gcn_id_fallback = id_fallback & current_t0["id_prefix"].eq("GCN")
ep_id_fallback = id_fallback & current_t0["id_prefix"].eq("EP")
grb_id_fallback = id_fallback & current_t0["id_prefix"].eq("GRB")
pending_gcn_trigger_time = (
    missing_real_t0
    & current_t0["anchor_type"].eq("trigger")
    & current_t0["t0_from_id"].isna()
)
pending_first_detection = (
    missing_real_t0 & current_t0["anchor_type"].eq("first_detection")
)
created_at_fallback = missing_real_t0 & ~id_fallback

id_p95_abs_hours = (
    id_validation.groupby("id_prefix")["abs_delta_seconds"].quantile(0.95) / 3600.0
)
id_max_abs_hours = (
    id_validation.groupby("id_prefix")["abs_delta_seconds"].max() / 3600.0
)
created_max_hours = created_validation.groupby("anchor_type")["delta_hours"].max()

ep_population_check_n = int((sub["id_prefix"] == "EP").sum())
ep_status = "PROVISIONAL — pending photometry causality check"
ep_validated_against = (
    f"{int((id_validation['id_prefix'] == 'EP').sum())} real t0 values; "
    "photometry causality check pending"
)

ladder = pd.DataFrame(
    [
        {
            "tier": "SkyPortal t0",
            "rule": "Use the populated listing t0",
            "n_sources_it_could_serve": int(current_t0["has_t0"].sum()),
            "measured_uncertainty": "direct field; 0 h assigned",
            "validated_against": "listing value",
            "status": "PHASE MATCHING",
        },
        {
            "tier": "GCN internal ID",
            "rule": "Parse GCN-YYMMDD_HHMMSS",
            "n_sources_it_could_serve": int(gcn_id_fallback.sum()),
            "measured_uncertainty": (
                f"p95 abs {id_p95_abs_hours['GCN']:.3f} h; "
                f"max abs {id_max_abs_hours['GCN']:.3f} h"
            ),
            "validated_against": f"{int((id_validation['id_prefix'] == 'GCN').sum())} real t0 values",
            "status": "PHASE MATCHING",
        },
        {
            "tier": "EP internal ID",
            "rule": "Parse EP-YYMMDD_HHMMSS",
            "n_sources_it_could_serve": int(ep_id_fallback.sum()),
            "measured_uncertainty": (
                f"p95 abs {id_p95_abs_hours['EP']:.3f} h; "
                f"max abs {id_max_abs_hours['EP']:.3f} h"
            ),
            "validated_against": ep_validated_against,
            "status": ep_status,
        },
        {
            "tier": "GRB internal ID",
            "rule": "Parse GRB-YYMMDD_HHMMSS",
            "n_sources_it_could_serve": int(grb_id_fallback.sum()),
            "measured_uncertainty": (
                f"p95 abs {id_p95_abs_hours['GRB']:.3f} h; "
                f"max abs {id_max_abs_hours['GRB']:.3f} h"
            ),
            "validated_against": f"{int((id_validation['id_prefix'] == 'GRB').sum())} real t0 values",
            "status": "DOSSIER ONLY; OUTLIER",
        },
        {
            "tier": "GCN TRIGGER_TIME",
            "rule": "Extract trigger time from associated circulars",
            "n_sources_it_could_serve": int(pending_gcn_trigger_time.sum()),
            "measured_uncertainty": "PENDING",
            "validated_against": "future GCN span-store step",
            "status": "PENDING",
        },
        {
            "tier": "First detection",
            "rule": "Use earliest validated survey detection",
            "n_sources_it_could_serve": int(pending_first_detection.sum()),
            "measured_uncertainty": "PENDING",
            "validated_against": "future full photometry ingestion",
            "status": "PENDING",
        },
        {
            "tier": "SkyPortal created_at",
            "rule": "Use row creation time as knowledge-time floor",
            "n_sources_it_could_serve": int(created_at_fallback.sum()),
            "measured_uncertainty": (
                f"trigger max {created_max_hours['trigger']:.3f} h; "
                f"first-detection max {created_max_hours['first_detection']:.3f} h"
            ),
            "validated_against": f"{len(created_validation)} real t0 values",
            "status": "DOSSIER ONLY",
        },
    ]
)
print(ladder.to_string(index=False))
current_t0["t0_source"] = "created_at_dossier_only"
current_t0["t0_uncertainty_hours"] = current_t0["anchor_type"].map(
    created_max_hours
)
current_t0.loc[current_t0["has_t0"], "t0_source"] = "skyportal_t0"
current_t0.loc[current_t0["has_t0"], "t0_uncertainty_hours"] = 0.0
for prefix, mask in [
    ("GCN", gcn_id_fallback),
    ("EP", ep_id_fallback),
    ("GRB", grb_id_fallback),
]:
    current_t0.loc[mask, "t0_source"] = f"source_id_timestamp_{prefix.lower()}"
    current_t0.loc[mask, "t0_uncertainty_hours"] = id_max_abs_hours[prefix]
current_t0["tier_status"] = "dossier_only"
current_t0.loc[current_t0["has_t0"], "tier_status"] = "phase_matching"
current_t0.loc[gcn_id_fallback, "tier_status"] = "phase_matching"
current_t0.loc[ep_id_fallback, "tier_status"] = "provisional"

selected_anchor_mjd = pd.to_numeric(current_t0["t0"], errors="coerce")
selected_anchor_mjd = selected_anchor_mjd.where(
    selected_anchor_mjd.notna(), current_t0["t0_from_id"]
)
selected_anchor_mjd = selected_anchor_mjd.where(
    selected_anchor_mjd.notna(), created_at_mjd
)
anchorless_count = int(selected_anchor_mjd.isna().sum())

selected_tiers = (
    current_t0["t0_source"].value_counts().rename_axis("selected_tier")
    .reset_index(name="sources")
)
print("Applied available tiers:")
print(selected_tiers.to_string(index=False))
print(f"Sources with no anchor at all: {anchorless_count}")
status_counts = (
    current_t0["tier_status"].value_counts()
    .reindex(["phase_matching", "provisional", "dossier_only"], fill_value=0)
    .rename_axis("tier_status")
    .reset_index(name="sources")
)
print("Status totals over all 800 sources:")
print(status_counts.to_string(index=False))

extra_columns = [
    "anchor_type", "t0_from_id", "t0_source", "t0_uncertainty_hours",
    "tier_status", "id_timestamp", "delta_id_created_s",
]
kept_base_columns = [
    column for column in source_index_base_columns if column not in extra_columns
]
source_index = current_t0[kept_base_columns + extra_columns].copy()
source_index.to_csv(EVIDENCE_DIR / "01_source_index.csv", index=False)


                tier                                           rule  n_sources_it_could_serve                                   measured_uncertainty                                    validated_against                                           status
        SkyPortal t0                   Use the populated listing t0                       128                             direct field; 0 h assigned                                        listing value                                   PHASE MATCHING
     GCN internal ID                        Parse GCN-YYMMDD_HHMMSS                       102                       p95 abs 0.057 h; max abs 0.364 h                                    53 real t0 values                                   PHASE MATCHING
      EP internal ID                         Parse EP-YYMMDD_HHMMSS                       189                       p95 abs 0.221 h; max abs 0.269 h 5 real t0 values; photometry causality check pending PROVISIONAL — pending photometry causality ch

**FINDING.** Applying the available tiers yields **230 phase-matching**, **189 provisional**, and **381 dossier-only** sources. All 800 receive a timestamp, but the 307 `created_at` fallbacks are knowledge-time anchors only.

### 8.7. DECISION

**DECISION.** **230 of 800 sources can enter phase matching, 189 are provisional
pending validation, and 381 are dossier-only.** The event table needs `t0`,
`t0_source`, and `t0_uncertainty_hours`, plus `anchor_type` and `tier_status`.

- **What the ID encodes.** The internal ID's `YYMMDD_HHMMSS` is generated at, or
  within seconds of, row-creation time by the ingestion pipeline. It is not copied
  from `created_at`, but when a record is created late the ID drifts with it. Across
  89 direct controls, the median absolute ID offset is 2 s versus 111 s for
  `created_at`, while the GRB outlier is late by 287 h despite a normal 63 s
  ID-to-creation gap.
- **GCN enters phase matching.** The **102** GCN ID fallbacks are supported by **53**
  direct controls, with a maximum absolute error of **0.364 h**.
- **EP is provisional.** The ID-versus-creation similarity metric is blind to the
  late-record failure mode (**8.5.5**). The EP and GCN bulk distributions are
  comparable, but their p95 tails differ by **18x** and risk lives in the tail
  (**8.5.2**). With zero failures in only five direct EP controls, the 95% upper bound
  on EP's true failure rate is approximately **45%**, while GRB is rejected on a
  measured **3.2%** failure rate (1 of 31); rejecting a measured failure while
  accepting an undetectable one would be inconsistent. Real `t0` covers only **2.6%**
  of EP sources versus **34.2%** of GCN sources, a **13x** difference that weakens the
  premise that both prefixes share the same ingestion mechanism.
- **GRB remains dossier-only.** One of 31 direct controls is wrong by **287.349 h**,
  and the failure cannot be distinguished from the other 73 GRB fallbacks using the
  listing alone.
- **What resolves EP.** A late-created record is detectable once per-source
  photometry is ingested: if the ID-derived `t0` is correct, no photometry point may
  have `mjd < t0`. A record created 12 days late produces detections earlier than its
  own claimed trigger -- a causality violation. This check applies to all **194 EP
  sources**, not to 5, and arrives with the per-source download stage.

The pending GCN `TRIGGER_TIME` and first-detection tiers remain unresolved. Until they
are measured, `created_at` serves 307 sources only as a dossier knowledge-time anchor.
`spectrum_exists` is true for **1 of 800** sources, and non-empty SkyPortal
`annotations` appear on **3 of 982** raw listing records; both remain residual fact
types rather than design anchors.

## 9. Decisions summary

**Question.** Which decisions are established by this notebook, and where is their
evidence?

In [19]:
decisions = pd.DataFrame(
    [
        ("Freeze dated evidence", "All counts tied to 2026-07-20", "0"),
        ("Population is the four-profile union", "982 records, 800 unique IDs", "1, 2"),
        ("Deduplicate with the richest profile record", "Only host fields differ; scientific arrays agree", "3"),
        ("Retain all listing sources", "224 absent from compact; 196 carry checked data flags", "4"),
        ("Separate fact inputs, raw context, and empty fields", "Full field coverage table", "5"),
        ("Reuse listing histories", "Four controls match per-source version counts", "6"),
        ("Download four missing row collections", "Collections absent; 3,200-call minimum scope", "7"),
        ("Separate trigger and first-detection anchors", "548 trigger anchors and 252 first-detection anchors", "8.2"),
        ("Accept directly supported phase-matching tiers", "128 real t0 values plus 102 validated GCN ID fallbacks", "8.3, 8.6"),
        ("Keep EP provisional pending a causality check", "189 EP ID fallbacks await full-photometry validation", "8.5, 8.7"),
        ("Keep weak fallbacks dossier-only", "74 GRB ID and 307 created_at fallbacks; 0 anchorless", "8.4, 8.6-8.7"),
    ],
    columns=["decision", "evidence", "section"],
)
print(decisions.to_string(index=False))

                                           decision                                               evidence      section
                              Freeze dated evidence                          All counts tied to 2026-07-20            0
               Population is the four-profile union                            982 records, 800 unique IDs         1, 2
        Deduplicate with the richest profile record       Only host fields differ; scientific arrays agree            3
                         Retain all listing sources  224 absent from compact; 196 carry checked data flags            4
Separate fact inputs, raw context, and empty fields                              Full field coverage table            5
                            Reuse listing histories          Four controls match per-source version counts            6
              Download four missing row collections           Collections absent; 3,200-call minimum scope            7
       Separate trigger and first-detect

**DECISION.** These decisions are the evidence-backed boundary for the next design
step. This notebook does not define the ledger schema; it establishes the population,
available source facts, required downloads, and unresolved trigger-time dependency.